# Agent 4 — Teams Alert Dispatcher

Sends Teams MessageCard alerts for CRITICAL and WARNING
KPI validation runs that have not yet been alerted.

Reads from:
  uc4_kpi_validation_run_b
  uc4_kpi_validation_results_b

Updates:
  uc4_kpi_validation_run_b (alert_sent, alert_sent_timestamp)

## Cell 2: Imports and Configuration

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

KPI_VALIDATION_RUN     = f"{CATALOG}.{SCHEMA}.uc4_kpi_validation_run_b"
KPI_VALIDATION_RESULTS = f"{CATALOG}.{SCHEMA}.uc4_kpi_validation_results_b"

TEAMS_WEBHOOK_URL = dbutils.secrets.get(
    scope="agent1-secrets",
    key  ="teams-webhook-url"
)

ALERT_CHANNEL     = "platform-support"
ALERT_ON_STATUSES = {"CRITICAL", "WARNING"}

STATUS_EMOJI = {
    "CRITICAL": "🔴",
    "WARNING" : "🟡",
    "PASS"    : "🟢"
}

THEME_COLOR = {
    "CRITICAL": "FF0000",
    "WARNING" : "FFA500",
    "PASS"    : "00FF00"
}

print("=" * 60)
print("AGENT 4 — TEAMS ALERT DISPATCHER")
print("=" * 60)
print(f"Alert channel  : {ALERT_CHANNEL}")
print(f"Webhook loaded : {'YES' if TEAMS_WEBHOOK_URL else 'NO'}")
print(f"Started at     : {datetime.now()}")
print("=" * 60)

## Cell 3: Load Runs to Alert

In [0]:
def load_runs_to_alert() -> list:
    """
    Loads KPI validation runs that need alerts.
    Conditions:
      1. Status is CRITICAL or WARNING
      2. AI enrichment is COMPLETED or NOT_REQUIRED
      3. Alert has not been sent
    """
    df = (
        spark.table(KPI_VALIDATION_RUN)
             .filter(
                 F.col("overall_status").isin(list(ALERT_ON_STATUSES)) &
                 F.col("ai_enrichment_status").isin(
                     ["COMPLETED", "NOT_REQUIRED"]
                 ) &
                 (
                     F.col("alert_sent").isNull() |
                     (F.col("alert_sent") == False)
                 )
             )
             .orderBy("business_date", "gold_fully_qualified_name")
    )

    runs = [row.asDict() for row in df.collect()]

    print(f"Runs requiring alerts: {len(runs)}")
    for r in runs:
        emoji = STATUS_EMOJI.get(r["overall_status"], "⚪")
        print(
            f"  {emoji} {r['business_date']}  "
            f"{r['gold_fully_qualified_name'].split('.')[-1]:<40}  "
            f"Status: {r['overall_status']}  "
            f"AI: {r['ai_enrichment_status']}"
        )

    return runs


runs_to_alert = load_runs_to_alert()

if not runs_to_alert:
    print("\nNo runs require alerts at this time.")
    NO_RUNS_TO_ALERT = True
else:
    NO_RUNS_TO_ALERT = False
    print(f"\nProceeding to dispatch {len(runs_to_alert)} alert(s).")

## Cell 4: Load Failed KPI Checks

In [0]:
def load_failed_kpi_checks(validation_run_id: str) -> list:
    """
    Loads WARNING and CRITICAL KPI checks for a run.
    Ordered by severity then KPI name.
    """
    df = (
        spark.table(KPI_VALIDATION_RESULTS)
             .filter(
                 (F.col("validation_run_id") == validation_run_id) &
                 (F.col("check_status").isin(["WARNING", "CRITICAL"]))
             )
             .orderBy(
                 F.when(F.col("check_status") == "CRITICAL", 1)
                  .otherwise(2),
                 "kpi_name",
                 "check_type"
             )
             .dropDuplicates(["kpi_name", "check_type"])
    )
    return [row.asDict() for row in df.collect()]


print("Load failed KPI checks function ready.")

## Cell 5: Build MessageCard

In [0]:
def format_kpi_check_line(check: dict) -> str:
    """
    Formats a single KPI check result as a readable line.
    """
    emoji      = STATUS_EMOJI.get(check["check_status"], "⚪")
    kpi_name   = check["kpi_name"]
    check_type = check["check_type"]
    observed   = check["observed_value"]
    expected   = check["expected_value"]
    deviation  = check["deviation_percentage"]

    if check_type == "FORMULA":
        detail = (
            f"{int(observed)} formula violation(s) detected "
            f"({deviation:.1f}% of rows)"
        )
    elif check_type == "CROSS_KPI":
        detail = (
            f"{int(observed)} logical violation(s) "
            f"({deviation:.1f}% of rows)"
        )
    elif check_type == "ANOMALY":
        direction = "above" if observed > expected else "below"
        detail = (
            f"observed {observed:.4f} vs baseline {expected:.4f} "
            f"({abs(deviation):.1f}% {direction} expected)"
        )
    elif check_type == "RANGE":
        detail = (
            f"{int(observed)} value(s) outside configured range "
            f"({deviation:.1f}% of rows)"
        )
    else:
        detail = f"observed {observed} expected {expected}"

    return (
        f"{emoji} **{kpi_name}** [{check_type}]  \n"
        f"{detail}"
    )


def build_agent4_message_card(
    run_record    : dict,
    failed_checks : list
) -> dict:
    """
    Builds a Teams MessageCard for an Agent 4 KPI validation alert.

    The card highlights which KPIs failed and why,
    with the AI summary explaining the business impact.
    """
    gold_fqn     = run_record["gold_fully_qualified_name"]
    table_name   = gold_fqn.split(".")[-1]
    biz_date     = str(run_record["business_date"])
    status       = run_record["overall_status"]
    run_id       = run_record["validation_run_id"]
    total_kpis   = run_record["total_kpis_checked"]
    kpis_passed  = run_record["total_kpis_passed"]
    kpis_warned  = run_record["total_kpis_warned"]
    kpis_critical= run_record["total_kpis_critical"]
    ai_status    = run_record["ai_enrichment_status"]

    emoji      = STATUS_EMOJI.get(status, "⚪")
    color      = THEME_COLOR.get(status, "0078D4")
    detected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

    # Get AI enrichment from first enriched check
    ai_summary    = None
    ai_action     = None
    ai_category   = None
    ai_confidence = None

    for check in failed_checks:
        if check.get("ai_summary"):
            ai_summary    = check["ai_summary"]
            ai_action     = check["ai_recommended_action"]
            ai_category   = check["ai_failure_category"]
            ai_confidence = check["ai_confidence_score"]
            break

    sections = []

    # ── Section 1: Header ──────────────────────────────────────────────────
    sections.append({
        "activityTitle"   : f"{emoji} {status} — Gold KPI Validation Alert",
        "activitySubtitle": "Agent 4 — Gold KPI Validation",
        "facts"           : [
            {"name": "Gold Table",    "value": table_name},
            {"name": "Full Name",     "value": gold_fqn},
            {"name": "Business Date", "value": biz_date},
            {"name": "Detected At",   "value": detected_at},
            {"name": "Overall Status","value": f"{emoji} {status}"}
        ],
        "markdown": True
    })

    # ── Section 2: KPI Summary ─────────────────────────────────────────────
    sections.append({
        "title": "📊 KPI Validation Summary",
        "facts": [
            {"name": "Total KPIs Checked", "value": str(total_kpis)},
            {"name": "✅ Passed",          "value": str(kpis_passed)},
            {"name": "🟡 Warning",         "value": str(kpis_warned)},
            {"name": "🔴 Critical",        "value": str(kpis_critical)}
        ],
        "markdown": True
    })

    # ── Section 3: Failed KPI Checks ──────────────────────────────────────
    if failed_checks:
        failed_lines = [
            format_kpi_check_line(check)
            for check in failed_checks
        ]

        sections.append({
            "title"   : f"❌ KPI Issues Found ({len(failed_checks)})",
            "text"    : "\n\n".join(failed_lines),
            "markdown": True
        })

    # ── Section 4: AI Analysis ─────────────────────────────────────────────
    if ai_summary:
        sections.append({
            "title"   : "🤖 AI KPI Analysis",
            "text"    : ai_summary,
            "markdown": True
        })

        if ai_action:
            sections.append({
                "title"   : "✅ Recommended Action",
                "text"    : ai_action,
                "markdown": True
            })

        sections.append({
            "title": "AI Enrichment Details",
            "facts": [
                {
                    "name" : "Failure Category",
                    "value": ai_category or "N/A"
                },
                {
                    "name" : "AI Confidence",
                    "value": f"{ai_confidence:.0%}" if ai_confidence else "N/A"
                }
            ],
            "markdown": True
        })

    # ── Section 5: Footer ──────────────────────────────────────────────────
    sections.append({
        "title": "🔍 Audit",
        "facts": [
            {
                "name" : "Validation Run ID",
                "value": run_id
            },
            {
                "name" : "Catalog.Schema",
                "value": f"{CATALOG}.{SCHEMA}"
            }
        ],
        "markdown": True
    })

    return {
        "@type"     : "MessageCard",
        "@context"  : "http://schema.org/extensions",
        "themeColor": color,
        "summary"   : f"Agent 4 Alert: {table_name} {status} on {biz_date}",
        "title"     : f"{emoji} {status}: {table_name} KPIs — {biz_date}",
        "sections"  : sections
    }


print("MessageCard builder ready.")

## Cell 6: Send and Mark Functions

In [0]:
def send_teams_alert(card: dict, run_id: str) -> bool:
    """Posts the card to Teams webhook."""
    try:
        response = requests.post(
            TEAMS_WEBHOOK_URL,
            headers={"Content-Type": "application/json"},
            data=json.dumps(card),
            timeout=30
        )
        if response.status_code in (200, 202):
            logger.info(f"Alert sent for run {run_id}")
            return True
        else:
            logger.error(
                f"Alert failed for run {run_id}. "
                f"Status: {response.status_code}"
            )
            return False
    except Exception as e:
        logger.error(f"Alert exception: {str(e)}")
        return False


def mark_alert_sent(run_id: str, success: bool):
    """Marks run as alerted to prevent duplicate alerts."""
    channel = ALERT_CHANNEL if success else "FAILED"
    spark.sql(f"""
        UPDATE {KPI_VALIDATION_RUN}
        SET
            alert_sent              = {str(success).lower()},
            alert_sent_timestamp    = '{datetime.now()}',
            alert_channel           = '{channel}'
        WHERE validation_run_id     = '{run_id}'
    """)


print("Send and mark functions ready.")

## Cell 7: Main Dispatch Loop

In [0]:
alerts_sent   = 0
alerts_failed = 0

if NO_RUNS_TO_ALERT:
    print("Skipping dispatch — no runs require alerts.")
else:
    print(f"Dispatching {len(runs_to_alert)} alert(s)...\n")

    for run_record in runs_to_alert:

        run_id     = run_record["validation_run_id"]
        gold_fqn   = run_record["gold_fully_qualified_name"]
        table_name = gold_fqn.split(".")[-1]
        biz_date   = run_record["business_date"]
        status     = run_record["overall_status"]
        emoji      = STATUS_EMOJI.get(status, "⚪")

        print(f"{'='*60}")
        print(f"{emoji} {table_name}  |  {biz_date}  |  {status}")
        print(f"{'='*60}")

        failed_checks = load_failed_kpi_checks(run_id)
        print(f"Failed KPI checks : {len(failed_checks)}")

        try:
            card    = build_agent4_message_card(run_record, failed_checks)
            success = send_teams_alert(card, run_id)
        except Exception as e:
            print(f"Card build error: {str(e)}")
            success = False

        mark_alert_sent(run_id, success)

        if success:
            alerts_sent += 1
            print(f"Alert sent ✅")
        else:
            alerts_failed += 1
            print(f"Alert failed ❌")

        print()

    print(f"Dispatch complete.")
    print(f"  Sent   : {alerts_sent}")
    print(f"  Failed : {alerts_failed}")

## Cell 8: Verify Alert Status

In [0]:
print("Alert status after dispatch:\n")
spark.sql(f"""
    SELECT
        business_date,
        SUBSTRING_INDEX(gold_fully_qualified_name, '.', -1) AS gold_table,
        overall_status,
        ai_enrichment_status,
        CASE WHEN alert_sent = true THEN 'YES' ELSE 'NO' END AS alert_fired,
        alert_sent_timestamp,
        alert_channel
    FROM {KPI_VALIDATION_RUN}
    ORDER BY business_date, gold_fully_qualified_name
""").show(20, truncate=False)

## Cell 9: Final Summary

In [0]:
print("\n" + "=" * 60)
print("AGENT 4 ALERT DISPATCHER COMPLETE")
print("=" * 60)

if NO_RUNS_TO_ALERT:
    print("Alerts sent   : 0 (no runs required alerting)")
else:
    print(f"Alerts sent   : {alerts_sent}")
    print(f"Alerts failed : {alerts_failed}")
    print(f"Channel       : {ALERT_CHANNEL}")

print(f"Completed at  : {datetime.now()}")
print("=" * 60)